# EEG Motor Imagery BCI — Signal & Pattern Visualizations

This notebook provides visual comparisons of pre-filtered vs. bandpass-filtered signals, spectral power density overlays, event-related desynchronization (ERD) scalp maps, and spatial filters.


In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from utils.preprocessing import load_and_preprocess
from utils.pipeline import get_run_pair
from utils.csp import CSP
from visualize_erd import compute_erd


## 1. Raw vs. Bandpass-Filtered EEG Signal
We filter Subject 1 raw data to the mu and beta bands (8-30 Hz) and plot a 10-second window across 5 motor cortex channels to observe the clean waveforms.


In [ ]:
subject = 1
runs = [3, 4]

# Load and standardize
edf_files = mne.datasets.eegbci.load_data(subject, runs, verbose='WARNING')
raws = [mne.io.read_raw_edf(f, preload=True, verbose='WARNING') for f in edf_files]
raw_pre = mne.concatenate_raws(raws)
mne.datasets.eegbci.standardize(raw_pre)
montage = mne.channels.make_standard_montage('standard_1005')
raw_pre.set_montage(montage)

# Apply bandpass filter
raw_filtered = raw_pre.copy().filter(8., 30., fir_design='firwin', verbose='WARNING')

# Crop to first 10 seconds
target_channels = ['C3', 'Cz', 'C4', 'FC3', 'FC4']
raw_plot_pre = raw_pre.copy().crop(0.0, 10.0)
raw_plot_post = raw_filtered.copy().crop(0.0, 10.0)

data_pre, times_pre = raw_plot_pre.get_data(picks=target_channels, return_times=True)
data_post, times_post = raw_plot_post.get_data(picks=target_channels, return_times=True)

fig, axes = plt.subplots(len(target_channels), 2, figsize=(15, 8), sharex=True)
for i, ch in enumerate(target_channels):
    # Pre-filter
    axes[i, 0].plot(times_pre, data_pre[i] * 1e6, color='#2c3e50')
    axes[i, 0].set_ylabel(f"{ch} (μV)")
    axes[i, 0].grid(True, linestyle='--', alpha=0.5)
    if i == 0:
        axes[i, 0].set_title("Raw EEG Signal", fontweight='bold')
    
    # Post-filter
    axes[i, 1].plot(times_post, data_post[i] * 1e6, color='#16a085')
    axes[i, 1].grid(True, linestyle='--', alpha=0.5)
    if i == 0:
        axes[i, 1].set_title("Bandpass Filtered EEG (8-30 Hz)", fontweight='bold')

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## 2. Power Spectral Density (PSD) Overlay
We calculate the power spectral density before and after bandpass filtering using Welch's method.


In [ ]:
sfreq = raw_pre.info['sfreq']
data_pre_all = raw_pre.get_data()
data_post_all = raw_filtered.get_data()

freqs, psd_pre = welch(data_pre_all, fs=sfreq, nperseg=int(2*sfreq))
_, psd_post = welch(data_post_all, fs=sfreq, nperseg=int(2*sfreq))

mean_psd_pre = 10 * np.log10(np.mean(psd_pre, axis=0))
mean_psd_post = 10 * np.log10(np.mean(psd_post, axis=0))

plt.figure(figsize=(10, 5))
plt.plot(freqs, mean_psd_pre, label='Raw (Pre-filter)', color='#e74c3c', alpha=0.7)
plt.plot(freqs, mean_psd_post, label='Filtered (8-30 Hz)', color='#2ecc71', alpha=0.9)
plt.xlim(1, 50)
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (dB/Hz)')
plt.title('Power Spectral Density (PSD) Pre vs. Post Filtering')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 3. Event-Related Desynchronization (ERD) Scalp Maps
We compute the relative power change between imagery tasks and the baseline rest state (T0) to show contralateral desynchronization.


In [ ]:
print("Computing ERD topographic maps for Subject 1...")
erd_left, info = compute_erd(subject, [4, 8, 12], 'T1')
erd_right, _ = compute_erd(subject, [4, 8, 12], 'T2')
erd_feet, _ = compute_erd(subject, [6, 10, 14], 'T2')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
max_val = max(np.max(np.abs(erd_left)), np.max(np.abs(erd_right)), np.max(np.abs(erd_feet)))
vmin, vmax = -max_val, max_val
if max_val < 15:
    vmin, vmax = -15, 15

# Plots
im_left, _ = mne.viz.plot_topomap(erd_left, info, axes=axes[0], show=False, cmap='RdBu_r', vlim=(vmin, vmax))
axes[0].set_title("Imagining Left Hand\n(Desynchronization at Right C4)", fontsize=10, fontweight='bold')

im_right, _ = mne.viz.plot_topomap(erd_right, info, axes=axes[1], show=False, cmap='RdBu_r', vlim=(vmin, vmax))
axes[1].set_title("Imagining Right Hand\n(Desynchronization at Left C3)", fontsize=10, fontweight='bold')

im_feet, _ = mne.viz.plot_topomap(erd_feet, info, axes=axes[2], show=False, cmap='RdBu_r', vlim=(vmin, vmax))
axes[2].set_title("Imagining Both Feet\n(Desynchronization at Central Cz)", fontsize=10, fontweight='bold')

cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im_left, cax=cbar_ax)
cbar.set_label('ERD / ERS (% change in mu/beta power)')
fig.suptitle(f"Event-Related Desynchronization (ERD) Topographic Maps (Subject {subject})", fontsize=12, fontweight='bold', y=1.05)
plt.show()


## 4. Custom CSP Spatial Filters
We fit our custom CSP algorithm on preprocessed epochs to construct the topographic maps of the spatial filters.


In [ ]:
X, y = load_and_preprocess(subject, [3, 4])
csp = CSP(n_components=4)
csp.fit(X, y)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i in range(4):
    mne.viz.plot_topomap(csp.filters_[i], info, axes=axes[i], show=False)
    axes[i].set_title(f"Component {i+1}", fontsize=10, fontweight='bold')
fig.suptitle("Custom CSP Spatial Filters Scalp Topographies", fontsize=12, fontweight='bold')
plt.show()
